In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma

In [3]:
## 1. Load the text document (Data Ingestion)
text_loader = TextLoader('speech.txt')
text_doc = text_loader.load()

In [4]:
## 2. Split the text document into smaller chunks (Data Transformation)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10,chunk_overlap=5)
split_docs = text_splitter.split_documents(text_doc)

In [5]:
## 3. Create embeddings for the text chunks along with vector store (Data Storage)
embeddings = OllamaEmbeddings()
vector_data_base = Chroma.from_documents(documents = split_docs, embedding = embeddings)

C:\Users\MohamedHafez\AppData\Local\Temp\ipykernel_23444\3040981692.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings()


In [ ]:
## 4. Query the vector store (Data Retrieval)
query = "What is the importance of managing our time?"
response = vector_data_base.similarity_search(query)

[Document(id='d5837691-5d57-4699-8ab5-e62fed88c372', metadata={'source': 'speech.txt'}, page_content='resource—'),
 Document(id='ddd565e1-e4dc-430f-9423-39ad2c15e25f', metadata={'source': 'speech.txt'}, page_content='Let’s'),
 Document(id='7414b503-c1cf-459a-8c09-df4755b8f1bc', metadata={'source': 'speech.txt'}, page_content='best time'),
 Document(id='cbbb43ef-9127-44a1-aac8-f00578e49d08', metadata={'source': 'speech.txt'}, page_content='time to')]

In [9]:
## Save to local disk
vector_data_base = Chroma.from_documents(documents = split_docs, embedding = embeddings, persist_directory="./chroma_db")

In [10]:
## Load from local disk
reloaded_vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
docs = reloaded_vector_db.similarity_search(query)
print(docs[0].page_content)

resource—


In [11]:
response_with_score = vector_data_base.similarity_search_with_score(query)
response_with_score

[(Document(id='b6e2091a-566f-4a2b-8cd0-6235bfd0a076', metadata={'source': 'speech.txt'}, page_content='resource—'),
  19110.0390625),
 (Document(id='25fd3ffc-8da8-47b3-95d5-7a57271202d3', metadata={'source': 'speech.txt'}, page_content='Let’s'),
  19753.716796875),
 (Document(id='6c65df01-5fe8-4578-9ded-beb65ff9fb38', metadata={'source': 'speech.txt'}, page_content='time to'),
  19833.16015625),
 (Document(id='447a2325-5993-4a44-8a45-d60f1ee3194b', metadata={'source': 'speech.txt'}, page_content='best time'),
  19845.466796875)]

In [12]:
retriever = vector_data_base.as_retriever()
retriever.invoke(query)

[Document(id='b6e2091a-566f-4a2b-8cd0-6235bfd0a076', metadata={'source': 'speech.txt'}, page_content='resource—'),
 Document(id='25fd3ffc-8da8-47b3-95d5-7a57271202d3', metadata={'source': 'speech.txt'}, page_content='Let’s'),
 Document(id='6c65df01-5fe8-4578-9ded-beb65ff9fb38', metadata={'source': 'speech.txt'}, page_content='time to'),
 Document(id='447a2325-5993-4a44-8a45-d60f1ee3194b', metadata={'source': 'speech.txt'}, page_content='best time')]